실습 5. 전처리 파이프라인 함수
- 불러오기·정규화·시간 보간을 한 함수로 묶어 자동화

목표
- 불러오기·정규화·시간 보간을 한 함수로 묶어 전처리를 자동화

단계
- 파일을 시간 인덱스로 만들고 격자로 정규화하는 함수 작성
- 함수 안에서 시간 보간으로 결측을 채우기
- 처리 전후 결측을 함께 출력해 검증

예상 결과
- 처리 전 각 22칸, 처리 후 0; 한 번의 호출로 깨끗한 데이터

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")

# 한글깨짐 해결
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv('../data/22_열처리.csv', encoding='utf-8')

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp').sort_index()
norm = df[['제어출력', '소입로온도']].asfreq('10s')

In [ ]:
# [재사용성을 확보한 모듈화된 전처리 파이프라인 함수 설계]
# 1. def preprocess(path, freq='10s'): CSV 파일 경로와 타겟 주기를 매개변수(Parameter)로 받아 분석 전 과정을 추상화합니다.
# 2. 데이터 수신 -> datetime 타입 변환 -> 시간 인덱싱 및 정렬 -> asfreq 격자 정규화 -> 시간 보간 완료 후 깨끗한 데이터프레임 반환.
# * 공장에서 센서 CSV 파일이 매일 아침 새로 수집되는 상황에서, 매번 타이핑하지 않고
#   정형화된 전처리 함수(preprocess)에 파일 경로만 입력해 결측과 주기를 일괄 정제하여 활용합니다.
# * 함수 내에서 계산 결과를 외부로 내보내기 위해 `return` 구문을 빠트리지 않는 것이 매우 중요합니다.

def preprocess(path, freq='10s'):
    d = pd.read_csv(path)
    d['timestamp'] = pd.to_datetime(d['timestamp'])
    d = d.set_index('timestamp').sort_index()
    g = d[['제어출력', '소입로온도']].asfreq(freq)

    print('전:', g.isna().sum().to_dict())            # 22, 22
    clean = g.interpolate(method='time')
    print('후:', clean.isna().sum().to_dict())         # 0, 0

    return clean

clean = preprocess('../data/22_열처리.csv')